# 1. WikiWhy: данные и постановка

Просмотр структуры WikiWhy и подготовка формата диалога в `data.py`. Ячейки используют стандартную библиотеку и читают существующие файлы без загрузки данных, токенизатора или модели.

Данные загружаются отдельной командой `python download_data.py`. Обучение выполняется в `02_training.ipynb` на GPU-сервере.

In [1]:
import json
from pathlib import Path
from collections import Counter
from pprint import pprint

ROOT = Path.cwd()
assert (ROOT / 'download_data.py').is_file(), 'Рабочая папка notebook должна быть hw4_lora'
DATA_DIR = ROOT / 'data'
assert (DATA_DIR / 'wikiwhy/train.jsonl').is_file(), 'Данных пока нет; загрузка выполняется отдельно'

raw = {}
for split in ('train', 'validation', 'test'):
    path = DATA_DIR / 'wikiwhy' / f'{split}.jsonl'
    raw[split] = [json.loads(line) for line in path.read_text().splitlines()]
history_path = DATA_DIR / 'mmlu_history/test.jsonl'
history_test = [json.loads(line) for line in history_path.read_text().splitlines()] if history_path.exists() else []
manifest_path = DATA_DIR / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    pprint({'wikiwhy': manifest['wikiwhy'], 'mmlu_history': manifest['mmlu_history']})
else:
    print('Режим просмотра WikiWhy: полный manifest и внешний тест появятся после загрузки на сервере.')

Режим просмотра WikiWhy: полный manifest и внешний тест появятся после загрузки на сервере.


## Объём и темы

Распределение примеров WikiWhy по split и темам. Для Full FT и LoRA используются одинаковые данные и исходные split.

In [ ]:
for split, rows in raw.items():
    print(split, 'всего:', len(rows), 'History:', sum(row['topic'] == 'History' for row in rows))
print('MMLU-Pro/history:', len(history_test))
pprint(Counter(row['topic'] for row in raw['train']).most_common())

## Одна строка и типы полей

`question` — why-вопрос; `cause` — короткий ответ; `explanation` — объяснение связи в виде строки. `ctx` — исходный фрагмент статьи, `effect` — следствие, `title/topic/split/id` — сведения о примере. У некоторых строк `topic` отсутствует (`None`).

По заданному формату входом служит только вопрос. Поля `cause` и `explanation` используются для эталонного ответа, `ctx` остаётся для просмотра источника.

In [ ]:
example = next(row for row in raw['train'] if row['topic'] == 'History')
pprint({key: type(value).__name__ for key, value in example.items()})
for key in ('id', 'topic', 'title', 'question', 'cause', 'explanation'):
    print()
    print(f'{key}:')
    print(example[key])

In [ ]:
# Примеры для обоснования датасета в отчёте.
examples = [row for row in raw['train'] if row['topic'] == 'History'][:3]
for row in examples:
    print()
    print('Вопрос:', row['question'])
    print('Причина:', row['cause'])
    print('Объяснение:', row['explanation'])

## Внешний исторический экзамен

MMLU-Pro/history используется только при оценке. Модель получает `question` и `options`. Поля `answer`, `answer_index` и `cot_content` содержат эталон и не входят в prompt. Число вариантов определяется конкретным вопросом.

In [ ]:
if history_test:
    exam = history_test[0]
    pprint({key: type(value).__name__ for key, value in exam.items()})
    print(exam['question'])
    pprint(exam['options'])
    print('Эталон для проверки:', exam['answer'])
else:
    print('MMLU-Pro/history здесь не загружен. Эта ячейка покажет пример после загрузки на сервере.')

## Формат диалога и эталона

`make_prompt` и `make_target` из `data.py` формируют сообщения модели и эталонный ответ в формате Answer/Explanation.

In [ ]:
%load_ext autoreload
%autoreload 2
from data import make_prompt, make_target

messages = make_prompt(example)
target = make_target(example)
pprint(messages)
print(target)

## Обучение и оценка

`02_training.ipynb` содержит подготовку тензоров и обучение Full FT и LoRA. `03_evaluation_report.ipynb` содержит расчёт метрик и сравнение результатов.